In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# RSNA Knee — Step 4: Pseudo-Label Extraction from Reports
# Run this on CPU (no GPU needed)
# ============================================================
# This extracts binary labels from free-text radiology reports
# for the 4,349 unlabeled studies. These pseudo-labels are
# noisy but provide enough signal to pretrain the CNN.
# ============================================================

import re
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
BASE     = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TRAIN_CSV = BASE / "train.csv"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"
]

# ── Negation detector ────────────────────────────────────────
# If any of these words appear within 6 words BEFORE a finding,
# treat the finding as ABSENT (negative).
# Add to NEGATION_TERMS list:
NEGATION_TERMS = [
    r"no\b", r"not\b", r"without\b", r"intact\b", r"normal\b",
    r"negative\b", r"unremarkable\b", r"keine\b", r"kein\b",
    r"sin\b", r"no\s+se\s+observa",
    r"pas\s+de", r"absence\s+de",
    r"grade\s+[I1]\b",      # ← ADD: grade I = not a real finding
    r"grade\s+[I1][I1]\b",  # ← ADD: grade II = minor, exclude
]

In [ ]:
def has_negation(text, match_start, window=60):
    """Check if a negation term appears within `window` chars before the match."""
    preceding = text[max(0, match_start - window):match_start]
    for neg in NEGATION_TERMS:
        if re.search(neg, preceding, re.IGNORECASE):
            return True
    return False

# ── Per-target keyword patterns ──────────────────────────────
# Each entry: (target_name, list_of_regex_patterns)
# Patterns are tried in order; first match wins.
# Use word boundaries (\b) to avoid partial matches.

PATTERNS = {
    "ACL": [
        r"\bACL\b.*?(tear|rupture|injury|sprain|disruption|lesion|damage)",
        r"\banterior\s+cruciate\b.*?(tear|rupture|injury|sprain|disruption)",
        r"\bvorderes\s+kreuzband\b",           # German
        r"\bliga[mn]ento\s+cruzado\s+anterior\b.*?(rotura|lesion)",  # Spanish
    ],
    "MCL": [
        r"\bMCL\b.*?(tear|rupture|injury|sprain|disruption|lesion|damage)",
        r"\bmedial\s+collateral\b.*?(tear|rupture|injury|sprain|disruption)",
        r"\binnenbandl[äa]sion\b",             # German
    ],
    "Medial Meniscus": [
        r"\bmedial\s+meniscus\b.*?(tear|rupture|injury|lesion|damage|degenerat)",
        r"\bmedial\s+menisk\w*\b.*?(riss|l[äa]sion|verletzung)",  # German
        r"\bmenisco\s+medial\b.*?(rotura|lesion|desgarro)",        # Spanish
    ],
    "Lateral Meniscus": [
        r"\blateral\s+meniscus\b.*?(tear|rupture|injury|lesion|damage|degenerat)",
        r"\bau[sß]en\w*\s+menisk\w*\b.*?(riss|l[äa]sion)",       # German
        r"\bmenisco\s+lateral\b.*?(rotura|lesion|desgarro)",       # Spanish
    ],
    "Medial OA": [
        r"\bmedial\s+(compartment\s+)?(osteoarthritis|OA|arthrosis|arthritis)",
        r"\bmediale\s+(gonarthrose|arthrose)",
        r"\bartrosis\s+medial\b",
        r"\bmedial\s+(joint\s+space\s+)?narrowing\b",
        r"\bmedial\s+tibiofemoral\b.*?(OA|arthr|narrow)",
        # Common OA descriptors often used instead of the word OA:
        r"\bmedial\b.*?(osteophyte|chondral|cartilage\s+loss|subchondral|sclerosis|eburnation)",
        r"\bmedial\b.*?(joint\s+space)\b.*?(reduc|narrow|decreas|loss)",
        # Turkish
        r"\bmedial\b.*?(dejenerasyon|artrit|artrose|kıkırdak\s+kayb)",
        r"\biç\s+(kompartman|bölme)\b.*?(daralma|artrit|artrose)",
    ],
    "Lateral OA": [
        r"\blateral\s+(compartment\s+)?(osteoarthritis|OA|arthrosis|arthritis)",
        r"\blaterale\s+(gonarthrose|arthrose)",
        r"\bartrosis\s+lateral\b",
        r"\blateral\s+(joint\s+space\s+)?narrowing\b",
        r"\blateral\s+tibiofemoral\b.*?(OA|arthr|narrow)",
        # Common OA descriptors:
        r"\blateral\b.*?(osteophyte|chondral|cartilage\s+loss|subchondral|sclerosis|eburnation)",
        r"\blateral\b.*?(joint\s+space)\b.*?(reduc|narrow|decreas|loss)",
        # Turkish
        r"\blateral\b.*?(dejenerasyon|artrit|artrose|kıkırdak\s+kayb)",
        r"\bdış\s+(kompartman|bölme)\b.*?(daralma|artrit|artrose)",
    ],
    "PF OA": [
        r"\bpatellofemoral\b.*?(osteoarthritis|OA|arthrosis|arthritis|narrow|chondro)",
        r"\bPF\s+(OA|arthr)",
        r"\bretropatellar\b.*?(arthr|chondro|OA)",
        r"\bpatella\b.*?(chondromalacia|osteoarthritis|arthrosis)",
    ],
    "Effusion": [
        r"\b(joint\s+)?effusion\b",
        r"\bsynovial\s+fluid\b",
        r"\bfluid\s+(in|within)\s+(the\s+)?(joint|knee)",
        r"\bgelenkerguss\b",                    # German
        r"\bderrame\s+(articular|sinovial)\b",  # Spanish
        r"\b[ée]panchement\b",                  # French
    ],
    "Synovitis": [
        r"\bsynovitis\b",
        r"\bsynovial\s+(inflammation|thickening|hypertrophy)",
        r"\bsynoviale\s+(verdickung|entz[üu]ndung)\b",
        r"\bsinovitis\b",                       # Spanish/Portuguese
    ],
    "Baker's": [
        r"\bbaker[''s]*\s*cyst\b",
        r"\bpopliteal\s+cyst\b",
        r"\bbacke?r\s*zyste\b",                 # German
        r"\bquiste\s+(popliteo|de\s+baker)\b",  # Spanish
        r"\bkyste\s+(poplite|de\s+baker)\b",    # French
    ],
    "Contusion": [
        r"\b(bone\s+)?(contusion|bruise|bruising)\b",
        r"\bbone\s+marrow\s+(edema|oedema|signal)\b",
        r"\bknochenmarködem\b",                 # German
        r"\bedema\s+(óseo|de\s+m[eé]dula)\b",  # Spanish
    ],
    "Fracture": [
        r"\bfracture\b",
        r"\bfraktur\b",                         # German
        r"\bfractura\b",                        # Spanish
        r"\bfracture\s+de\b",                   # French
        r"\bstress\s+fracture\b",
        r"\binsufficiency\s+fracture\b",
        r"\bkortikalisunterbrechung\b",          # German cortical break
    ],
}

In [ ]:
# ── Extraction function ──────────────────────────────────────
def extract_labels_from_report(report: str) -> dict:
    """
    Extract binary labels from one radiology report.
    Returns dict: {target: 0 or 1} for all 12 targets.
    NaN = could not determine (ambiguous / uncertain language).
    """
    if not isinstance(report, str) or len(report.strip()) == 0:
        return {t: np.nan for t in TARGETS}

    # Lowercase for matching but keep original for negation window
    text = report.lower()
    results = {}

    for target, patterns in PATTERNS.items():
        found = False
        for pattern in patterns:
            for m in re.finditer(pattern, text, re.IGNORECASE | re.DOTALL):
                if not has_negation(text, m.start()):
                    found = True
                    break
            if found:
                break
        results[target] = 1 if found else 0

    return results

In [ ]:
 # ── Run extraction ───────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv(TRAIN_CSV)
    print(f"Total studies: {len(df)}")

    # Identify unlabeled studies (no hard labels)
    has_labels = df[TARGETS].notna().any(axis=1)
    unlabeled  = df[~has_labels].copy()
    labeled    = df[has_labels].copy()
    print(f"Labeled  : {len(labeled)}")
    print(f"Unlabeled: {len(unlabeled)}  (will extract from reports)")

    # ── Extract pseudo-labels ────────────────────────────────
    print("\nExtracting pseudo-labels from reports...")
    pseudo_rows = []
    empty_reports = 0

    for _, row in unlabeled.iterrows():
        labels = extract_labels_from_report(row["Report"])
        labels["StudyInstanceUID"] = row["StudyInstanceUID"]
        labels["Report"]           = row["Report"]
        labels["is_pseudo"]        = True
        pseudo_rows.append(labels)

        if not isinstance(row["Report"], str):
            empty_reports += 1

    pseudo_df = pd.DataFrame(pseudo_rows)
    print(f"Done. Empty reports: {empty_reports}")

    # ── Stats ────────────────────────────────────────────────
    print("\n=== Pseudo-label positive rates ===")
    for t in TARGETS:
        pos   = pseudo_df[t].sum()
        total = pseudo_df[t].notna().sum()
        print(f"  {t:<20} {int(pos):>4} / {total}  ({100*pos/total:.1f}%)")

    print("\n=== Real label positive rates (for comparison) ===")
    labeled[TARGETS] = labeled[TARGETS].fillna(0)
    for t in TARGETS:
        pos   = labeled[t].sum()
        total = len(labeled)
        print(f"  {t:<20} {int(pos):>3} / {total}  ({100*pos/total:.1f}%)")

    # ── Save ─────────────────────────────────────────────────
    # Combine real + pseudo labels
    labeled["is_pseudo"] = False
    combined = pd.concat([
        labeled[["StudyInstanceUID", "Report", "is_pseudo"] + TARGETS],
        pseudo_df[["StudyInstanceUID", "Report", "is_pseudo"] + TARGETS]
    ], ignore_index=True)

    combined.to_csv("pseudo_labeled_train.csv", index=False)
    print(f"\n✅ Saved pseudo_labeled_train.csv — {len(combined)} studies total")
    print(f"   ({len(labeled)} real + {len(pseudo_df)} pseudo)")

    # ── Spot-check 3 reports ─────────────────────────────────
    print("\n=== SPOT CHECK (3 random reports) ===")
    sample = unlabeled[unlabeled["Report"].notna()].sample(3, random_state=42)
    for _, row in sample.iterrows():
        labels = extract_labels_from_report(row["Report"])
        positives = [t for t, v in labels.items() if v == 1]
        print(f"\nReport snippet: {str(row['Report'])[:200]}...")
        print(f"Extracted positives: {positives}")

In [ ]:
# ============================================================
# RSNA Knee — Step 5: Pretrain on Pseudo-Labels
# Then fine-tune on 58 real labeled studies
# ============================================================
# TWO-STAGE TRAINING:
#   Stage 1: Train on ~4,349 pseudo-labeled studies (noisy)
#            Higher LR, more epochs, treat as pretraining
#   Stage 2: Fine-tune on 58 real labeled studies
#            Lower LR, fewer epochs, load Stage 1 weights
# ============================================================

import os, random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ── assumes KneeDataset, collate_fn, KneeModel, compute_auc,
#    train_one_epoch, validate are already defined (from step2) ──

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"
]

class CFG:
    seed        = 42
    img_size    = 224
    max_slices  = 16
    backbone    = "efficientnet_b2"
    pretrained  = True
    n_targets   = 12
    batch_size  = 1
    accum_steps = 4
    device      = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp     = True

    BASE        = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
    TRAIN_CSV   = BASE / "train.csv"
    SERIES_CSV  = BASE / "train_series.csv"
    TRAIN_DIR   = BASE / "train_series"
    PSEUDO_CSV  = Path("/kaggle/working/pseudo_labeled_train.csv")  # output of step4

    # Stage 1 — pretrain on pseudo-labels
    pretrain_epochs = 5     # keep short: data is noisy
    pretrain_lr     = 2e-4

    # Stage 2 — fine-tune on real labels
    finetune_epochs = 30
    finetune_lr     = 3e-5  # much lower: preserve pretrained features
    num_folds       = 3
    train_fold      = 0
    weight_decay    = 1e-5


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)


# ── Stage 1: Pretrain ────────────────────────────────────────
def pretrain(model, pseudo_df, series_df, cfg):
    print(f"\n{'='*50}")
    print(f"STAGE 1: Pretraining on {len(pseudo_df)} pseudo-labeled studies")
    print(f"{'='*50}")

    # Use ALL pseudo-labeled studies (no val split needed here)
    # We only care about learning features, not overfitting
    ds     = KneeDataset(pseudo_df, series_df, cfg.TRAIN_DIR, augment=True)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                        collate_fn=collate_fn, num_workers=2)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.pretrain_lr,
                                  weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=cfg.pretrain_epochs)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    for epoch in range(cfg.pretrain_epochs):
        loss = train_one_epoch(model, loader, optimizer,
                               criterion, cfg.device, scaler)
        scheduler.step()
        print(f"  Pretrain Epoch {epoch+1}/{cfg.pretrain_epochs}  loss={loss:.4f}")

    torch.save(model.state_dict(), "pretrained_backbone.pth")
    print("✅ Pretrained weights saved: pretrained_backbone.pth")
    return model


# ── Stage 2: Fine-tune ───────────────────────────────────────
def finetune(model, labeled_df, series_df, cfg):
    print(f"\n{'='*50}")
    print(f"STAGE 2: Fine-tuning on {len(labeled_df)} real labeled studies")
    print(f"{'='*50}")

    # 3-fold CV
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=cfg.num_folds, shuffle=True, random_state=cfg.seed)
    labeled_df = labeled_df.copy()
    labeled_df["fold"] = -1
    for fold, (_, val_idx) in enumerate(kf.split(labeled_df)):
        labeled_df.loc[val_idx, "fold"] = fold

    train_df = labeled_df[labeled_df["fold"] != cfg.train_fold]
    val_df   = labeled_df[labeled_df["fold"] == cfg.train_fold]
    print(f"Fold {cfg.train_fold}: train={len(train_df)}, val={len(val_df)}")

    train_ds = KneeDataset(train_df, series_df, cfg.TRAIN_DIR, augment=True)
    val_ds   = KneeDataset(val_df,   series_df, cfg.TRAIN_DIR, augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True,  collate_fn=collate_fn, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size,
                              shuffle=False, collate_fn=collate_fn, num_workers=2)

    # Lower LR for fine-tuning — preserve pretrained features
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.finetune_lr,
                                  weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=cfg.finetune_epochs)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    best_auc   = 0.0
    patience   = 12
    no_improve = 0

    for epoch in range(cfg.finetune_epochs):
        loss = train_one_epoch(model, train_loader, optimizer,
                               criterion, cfg.device, scaler)
        val_auc, val_preds, val_labels = validate(model, val_loader, cfg.device)
        scheduler.step()

        print(f"Epoch {epoch+1:02d}/{cfg.finetune_epochs}  "
              f"loss={loss:.4f}  val_AUC={val_auc:.4f}")

        if (epoch + 1) % 5 == 0:
            print("  Per-target:")
            compute_auc(val_preds, val_labels, verbose=True)

        if val_auc > best_auc:
            best_auc   = val_auc
            no_improve = 0
            torch.save(model.state_dict(),
                       f"best_finetuned_fold{cfg.train_fold}.pth")
            print(f"  ✅ Saved (AUC={best_auc:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stop at epoch {epoch+1}")
                break

    print(f"\nBest Fine-tune Val AUC: {best_auc:.4f}")
    return best_auc


# ── Main ─────────────────────────────────────────────────────
if __name__ == "__main__":
    df_train  = pd.read_csv(CFG.TRAIN_CSV)
    df_series = pd.read_csv(CFG.SERIES_CSV)

    # Real labeled studies
    labeled = df_train[df_train[TARGETS].notna().any(axis=1)].reset_index(drop=True)
    labeled[TARGETS] = labeled[TARGETS].fillna(0)
    labeled["is_pseudo"] = False
    print(f"Real labeled studies: {len(labeled)}")

    # Pseudo-labeled studies (from step4)
    pseudo = pd.read_csv(CFG.PSEUDO_CSV)
    pseudo = pseudo[pseudo["is_pseudo"] == True].reset_index(drop=True)
    # Drop studies where ALL labels are 0 (report had no findings — still useful)
    # Drop studies where Report is NaN (no report at all — not useful)
    pseudo = pseudo[pseudo["Report"].notna()].reset_index(drop=True)
    pseudo[TARGETS] = pseudo[TARGETS].fillna(0)
    print(f"Pseudo labeled studies: {len(pseudo)}")

    # ── Stage 1: Pretrain on pseudo-labels ──────────────────
    model = KneeModel().to(CFG.device)
    model = pretrain(model, pseudo, df_series, CFG)

    # ── Stage 2: Fine-tune on real labels ───────────────────
    # Load pretrained weights (already in model, but reload to be safe)
    model.load_state_dict(torch.load("pretrained_backbone.pth",
                                      map_location=CFG.device))
    best_auc = finetune(model, labeled, df_series, CFG)

    print(f"\n{'='*50}")
    print(f"FINAL RESULT: Val AUC = {best_auc:.4f}")
    print(f"Target: > 0.75 to confirm pseudo-labeling helps")
    print(f"{'='*50}")